# 5. Repensando el Overfitting

#### 5.1 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [9]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

ERROR: Error in parse(text = input): <text>:2:6: unexpected symbol
1: # primero establecer el Runtime de Python 3
2: from google.colab
        ^


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"




---



## 5.2 rpart  Canaritos

Se agregarán canaritos al dataset, se entrenará el arbol con los mejores hiperparámetros encontrados, y se analizará si los canaritos aparecen en algun split.

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

### 5.2.1  carga manual de hiperparámetros
Aqui debe cargar SU semilla primigenia y
<br> SUS mejores hiperparámetros que encontró para el ARBOL DE DECISION, ya sea por Grid Search o  Bayesian Optimization

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 700001

PARAM$rpart$cp <- 0.001
PARAM$rpart$minsplit <- 50
PARAM$rpart$minbucket <- 10
PARAM$rpart$maxdepth <- 10

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5200"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

la siguiente celda tarda 4 minutos en correr

In [ ]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [10]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_35a34efec139 
                      2

vaya a su Google Drive
<br> busque la carpeta **My Drive / labo1 / exp / exp5200**
<br> baje el archivo **arbol_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hechos en alguna de las nuevas variables canaritos



---



## 5.3 rpart  Canaritos desconfiados

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [11]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,837514,44.8,1473245,78.7,1473245,78.7
Vcells,1651007,12.6,172348029,1315.0,215426887,1643.6


In [12]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia y
<br> parametros de un@ alumn@ desconfiad@

In [13]:
PARAM <- list()
PARAM$semilla_primigenia <- 700001

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 2000
PARAM$rpart$minbucket <- 800
PARAM$rpart$maxdepth <- 6

In [14]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5300"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [15]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [16]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [17]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

la siguiente celda tarda 4 minutos en correr

In [18]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [19]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos_desconfiados.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_35a33ee27535 
                      2

vaya a su Google Drive
<br> busque la carpeta **My Drive /  dm / labo1 / exp5300**
<br> baje el archivo **arbol_canaritos_desconfiados.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader
<br> y dentro del .pdf busque splits hecho en alguna de las nuevas variables canaritos



---



## 5.4 rpart  Canaritos pruning

Se trabaja con la original clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [20]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,837491,44.8,1473245,78.7,1473245,78.7
Vcells,1650967,12.6,165518108,1262.9,215426887,1643.6


In [21]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [22]:
PARAM <- list()
PARAM$semilla_primigenia <- 700001

PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16  # deberia pohner 31, por velocidad en la clase va el 16

In [23]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5400"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [24]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [25]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [26]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

la siguiente celda corre en 12 minutos

In [27]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_ternaria=="BAJA+2", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_ternaria ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [28]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [29]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_35a37b805a6c 
                      2

In [30]:
# datos del futuro
dfuture <- dataset[foto_mes == 202109]

In [31]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuture,
  type= "prob"
)

In [32]:
# tabla prediccion
tb_prediccion <- as.data.table(list(
  "numero_de_cliente" = dfuture$numero_de_cliente,
  "prob"=prediccion[, "BAJA+2"]
))

In [33]:
# Decison
PARAM$prob_corte <-  PARAM$peso/ ( PARAM$peso + 39)

tb_prediccion[ , Predicted := ifelse( prob< PARAM$prob_corte, 0L, 1L) ]

In [34]:
# archivo para kaggle
archivo_kaggle <- "K5400_001.csv"

fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

In [35]:
# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-a"
arch <- paste( "-f", archivo_kaggle)

In [36]:
mensaje <- paste0("-m 'cp=", PARAM$rpart$cp,
  "  minsplit=", PARAM$rpart$minsplit,
  "  minbucket=", PARAM$rpart$minbucket,
  "  maxdepth=", PARAM$rpart$maxdepth,
  "  peso=", PARAM$peso,
  "'"
 )

In [37]:
linea <- paste( comando, competencia, arch, mensaje)

# este es el comando que correria desde el prompt de Linux
linea

[1] "kaggle competitions submit -c data-mining-inicial-2026-a -f K5400_001.csv -m 'cp=-1  minsplit=2  minbucket=1  maxdepth=16  peso=10'"

In [38]:
# ejecuto el comando
salida <- system(linea, intern=TRUE)
cat(salida)

Successfully submitted to Data Mining, Inicial 2026 A

vaya a su Google Drive
<br> busque la carpeta **My Drive /  labo1 / exp / exp5400**
<br> baje el archivo **stopping_at_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader




---



## 5.5 rpart  Canaritos pruning BINARIA

Pasamos a trabajar con una clase  Binaria


*   POS = { BAJA+1, BAJA+2 }
*   NEG = { CONTINUA }




ahora la probabilidad que devuelve el modelo es de POS,
<br> ya no es la de BAJA+2,
<br> ya no puedo cortar por ella
<br> debo cortar por cnatidad de envios !

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [39]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,843968,45.1,1966910,105.1,1966910,105.1
Vcells,1665150,12.8,374444655,2856.8,371437223,2833.9


In [40]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [41]:
PARAM <- list()
PARAM$semilla_primigenia <- 700001

PARAM$envios <- 9000
PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16 # deberia ser 31, por velocidad en clsae se baja  16

In [42]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp5500"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [43]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [44]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [45]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

In [46]:
# clase binaria
dtrain[, clase_binaria2 := ifelse( clase_ternaria=="CONTINUA", "NEG", "POS" ) ]
dtrain[, clase_ternaria := NULL ]

la siguiente celda corre en 11 minutos

In [47]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_binaria2=="POS", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_binaria2 ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [48]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [49]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

agg_record_35a32f8f3bb3 
                      2

In [50]:
# datos del futuro
dfuture <- dataset[foto_mes == 202109]

In [51]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuture,
  type= "prob"
)

In [52]:
# tabla prediccion
tb_prediccion <- as.data.table(list(
  "numero_de_cliente" = dfuture$numero_de_cliente,
  "prob"=prediccion[, "POS"]
))

In [53]:
# Decison
setorder( tb_prediccion, -prob )
tb_prediccion[ , Predicted := 0L ]
tb_prediccion[ seq(PARAM$envios), Predicted := 1L ]


In [54]:
# archivo para kaggle
archivo_kaggle <- "KBin5500_005.csv"

fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

In [55]:
# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-a"
arch <- paste( "-f", archivo_kaggle)

In [56]:
mensaje <- paste0("-m 'cp=", PARAM$rpart$cp,
  "  minsplit=", PARAM$rpart$minsplit,
  "  minbucket=", PARAM$rpart$minbucket,
  "  maxdepth=", PARAM$rpart$maxdepth,
  "  envios=", PARAM$envios,
  "  peso=", PARAM$peso,
  "'"
 )

In [57]:
linea <- paste( comando, competencia, arch, mensaje)

# este es el comando que correria desde el prompt de Linux
linea

[1] "kaggle competitions submit -c data-mining-inicial-2026-a -f KBin5500_005.csv -m 'cp=-1  minsplit=2  minbucket=1  maxdepth=16  envios=9000  peso=10'"

In [58]:
# ejecuto el comando
salida <- system(linea, intern=TRUE)
cat(salida)

Successfully submitted to Data Mining, Inicial 2026 A

vaya a su Google Drive
<br> busque la carpeta **My Drive /  labo1 / exp / exp5500**
<br> baje el archivo **stopping_at_canaritos.pdf**  a su laptop
<br> abra el .pdf con el Acrobat Reader




---

